# Homework: Data Leakage, Cross-Validation, and Tree-Based Models

In this homework, you will use the **Forest CoverType** dataset to predict forest cover type from topographic, hydrologic, soil, and wilderness-area variables.

The goals are to:

1. Identify and explain **data leakage**.
2. Compare a leaky model to a properly validated model.
3. Use cross-validation to tune either a **random forest** or an **XGBoost** model.
4. Fit a final model and evaluate it on a held-out test set.
5. Explain the model workflow clearly enough that someone else could trust the results.

The dataset comes from the UCI Machine Learning Repository and contains observations from forested areas in Colorado.

## 0. Setup

In [ ]:
# If needed, install packages before running the notebook:
# pip install pandas numpy matplotlib seaborn scikit-learn xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    make_scorer,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

RANDOM_STATE = 7
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid")

## 1. Load The Data

In [ ]:
cover_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/covtype/covtype.data.gz"

cover_names = [
    "elevation",
    "aspect",
    "slope",
    "horizontal_distance_to_hydrology",
    "vertical_distance_to_hydrology",
    "horizontal_distance_to_roadways",
    "hillshade_9am",
    "hillshade_noon",
    "hillshade_3pm",
    "horizontal_distance_to_fire_points",
]

cover_names += [f"wilderness_area_{i}" for i in range(1, 5)]
cover_names += [f"soil_type_{i}" for i in range(1, 41)]
cover_names += ["cover_type"]

cover_raw = pd.read_csv(cover_url, names=cover_names)

cover_labels = {
    1: "Spruce/Fir",
    2: "Lodgepole Pine",
    3: "Ponderosa Pine",
    4: "Cottonwood/Willow",
    5: "Aspen",
    6: "Douglas-fir",
    7: "Krummholz",
}

cover_data = cover_raw.copy()
cover_data["cover_type"] = cover_data["cover_type"].map(cover_labels)

# Use a smaller sample so the assignment runs quickly on laptops.
cover_data = cover_data.sample(n=15000, random_state=RANDOM_STATE).reset_index(drop=True)

cover_data.head()

In [ ]:
cover_data.info()

## 2. Explore The Response Variable

In [ ]:
class_balance = (
    cover_data["cover_type"]
    .value_counts()
    .rename_axis("cover_type")
    .reset_index(name="n")
)

class_balance["percent"] = class_balance["n"] / class_balance["n"].sum()
class_balance

**Question 1.** Is this a balanced classification problem? Which classes are most common, and which are least common?

Write your answer here:

>

## 3. What Is Data Leakage?

Data leakage happens when information reaches the model during training that would **not be available at prediction time**.

Examples include:

- A predictor created directly from the response variable.
- Preprocessing the full dataset before splitting into training and test sets.
- Using future information to predict the past.
- Allowing duplicate or near-duplicate observations to appear in both training and test data.
- Computing group, spatial, or temporal summaries using the full dataset before validation.

In this assignment, we will create an intentionally leaky variable so you can see how leakage can make a model look unrealistically good.

## 4. Create A Leaky Dataset

The variable below is not a real environmental predictor. It is created from the answer we are trying to predict.

In [ ]:
label_encoder = LabelEncoder()
cover_data["cover_type_code"] = label_encoder.fit_transform(cover_data["cover_type"])
class_names = label_encoder.classes_

leaky_data = cover_data.copy()
leaky_data["leaky_cover_code"] = leaky_data["cover_type_code"]

leaky_data[["cover_type", "cover_type_code", "leaky_cover_code"]].head()

**Question 2.** Why is `leaky_cover_code` an example of data leakage?

Write your answer here:

>

## 5. Split The Leaky Dataset

In [ ]:
X_leaky = leaky_data.drop(columns=["cover_type", "cover_type_code"])
y_leaky = leaky_data["cover_type_code"]

X_leaky_train, X_leaky_test, y_leaky_train, y_leaky_test = train_test_split(
    X_leaky,
    y_leaky,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_leaky,
)

## 6. Fit A Leaky Model

This model is allowed to use the leaky predictor. Its performance should look suspiciously strong.

In [ ]:
leaky_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), X_leaky_train.columns),
    ],
    remainder="drop",
)

leaky_pipeline = Pipeline(
    steps=[
        ("preprocessor", leaky_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_features=8,
            min_samples_leaf=5,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]
)

leaky_pipeline.fit(X_leaky_train, y_leaky_train)
leaky_pred = leaky_pipeline.predict(X_leaky_test)

leaky_results = pd.DataFrame({
    "workflow": ["Leaky workflow"],
    "accuracy": [accuracy_score(y_leaky_test, leaky_pred)],
    "kappa": [cohen_kappa_score(y_leaky_test, leaky_pred)],
    "macro_f1": [f1_score(y_leaky_test, leaky_pred, average="macro")],
})

leaky_results

**Question 3.** Why should you distrust the leaky model, even if the accuracy is very high?

Write your answer here:

>

## 7. Build The Proper Dataset

Now remove the leaky variable and restart the modeling process correctly.

In [ ]:
model_data = cover_data.drop(columns=["cover_type_code"])

X = model_data.drop(columns=["cover_type"])
y = label_encoder.transform(model_data["cover_type"])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train.shape, X_test.shape

**Question 4.** Why do we split the data before cross-validation, tuning, or final model evaluation?

Write your answer here:

>

## 8. Cross-Validation Folds

Cross-validation should be created from the **training data only**.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

cv

## 9. Choose Your Model

Choose **one** model:

- `"random_forest"`
- `"xgboost"`

Then explain why you chose that model.

**Question 5.** Which model did you choose, and why? Your reason can be based on interpretability, speed, expected performance, or what you want to practice.

Write your answer here:

>

## 10. Create The Preprocessor

The predictors are already numeric, including the wilderness-area and soil-type indicator variables. We will still include a median imputation step as a best practice.

In [ ]:
numeric_features = X_train.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
    ],
    remainder="drop",
)

preprocessor

## 11. Define Candidate Models

In [ ]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
    ]
)

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]
)

## 12. Create The Workflow

Complete this section by selecting the pipeline that matches your chosen model.

In [ ]:
# Hint: chosen_pipeline should be either rf_pipeline or xgb_pipeline.

chosen_pipeline = #TODO

## 13. Create A Small Tuning Grid

For homework, use a modest grid so the code finishes in a reasonable amount of time. You may expand the grid if your computer can handle it.

In [ ]:
# TODO: create the tuning grid.

param_grid = #TODO

## 14. Tune With Cross-Validation

We will tune using macro F1. Macro F1 gives each class equal weight, which is useful when some classes are much less common than others.

In [ ]:
macro_f1 = make_scorer(f1_score, average="macro")

search = GridSearchCV(
    estimator=chosen_pipeline,
    param_grid=param_grid,
    scoring={
        "accuracy": "accuracy",
        "macro_f1": macro_f1,
    },
    refit="macro_f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)

search.fit(X_train, y_train)

cv_results = pd.DataFrame(search.cv_results_)
cv_results.sort_values("rank_test_macro_f1").head()

**Question 6.** Which metric do you think is most useful for this dataset: accuracy, kappa, or F-measure? Explain your choice.

Write your answer here:

>

## 15. Select The Best Model

In [ ]:
print("Best CV macro F1:", search.best_score_)
print("Best parameters:")
search.best_params_

**Question 7.** What hyperparameters were selected for your model? Do the selected values suggest a simple model or a more flexible model?

Write your answer here:

>

## 16. Final Test Evaluation

Only use the test set once, after model selection is complete.

In [ ]:
final_model = search.best_estimator_
final_pred = final_model.predict(X_test)

proper_results = pd.DataFrame({
    "workflow": ["Proper workflow"],
    "accuracy": [accuracy_score(y_test, final_pred)],
    "kappa": [cohen_kappa_score(y_test, final_pred)],
    "macro_f1": [f1_score(y_test, final_pred, average="macro")],
})

proper_results

In [ ]:
cm = confusion_matrix(y_test, final_pred)

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names,
).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")

ax.set_title("Final test-set confusion matrix")
ax.set_xlabel("Predicted cover type")
ax.set_ylabel("Observed cover type")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

**Question 8.** Which classes does the model predict well? Which classes are most often confused?

Write your answer here:

>

## 17. Compare Leaky And Proper Results

In [ ]:
comparison_results = pd.concat([leaky_results, proper_results], ignore_index=True)
comparison_results

**Question 9.** Compare the leaky model results to the proper final model results. What does this comparison teach you about data leakage?

Write your answer here:

>

## 18. Simple Permutation Importance

This section gives a model-agnostic way to ask: how much worse does the model get if one predictor is randomly shuffled?

In [ ]:
baseline_f1 = f1_score(y_test, final_pred, average="macro")

importance_variables = [
    "elevation",
    "slope",
    "horizontal_distance_to_hydrology",
    "horizontal_distance_to_roadways",
    "horizontal_distance_to_fire_points",
]

importance_rows = []
rng = np.random.default_rng(RANDOM_STATE)

for variable in importance_variables:
    X_permuted = X_test.copy()
    X_permuted[variable] = rng.permutation(X_permuted[variable].to_numpy())

    permuted_pred = final_model.predict(X_permuted)
    permuted_f1 = f1_score(y_test, permuted_pred, average="macro")

    importance_rows.append({
        "variable": variable,
        "baseline_macro_f1": baseline_f1,
        "permuted_macro_f1": permuted_f1,
        "importance": baseline_f1 - permuted_f1,
    })

permutation_importance = (
    pd.DataFrame(importance_rows)
    .sort_values("importance", ascending=False)
)

permutation_importance

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(
    data=permutation_importance,
    x="importance",
    y="variable",
    color="steelblue",
)
plt.axvline(0, color="black", linewidth=1)
plt.title("Simple permutation importance")
plt.xlabel("Drop in macro F1 after shuffling")
plt.ylabel(None)
plt.tight_layout()
plt.show()

**Question 10.** Which variables appear most important in your model? Does that make environmental sense?

Write your answer here:

>

## 19. Final Reflection

Answer the following in a short paragraph.

**Question 11.** Describe your complete modeling workflow. Your answer should mention:

- How you avoided data leakage.
- Why you used cross-validation.
- Which model you chose.
- Which metric you used to select the model.
- How the final test performance compared to the leaky model.
- One limitation of this modeling approach.

Write your answer here:

>